In [1]:
from packages.transcripts import transcribe_with_timestamps
from packages.PII_detection import detect_pii_entities
from packages.mapTimeStamps import map_pii_to_timestamps, map_pii_to_timestamps_with_words
from packages.getFiles import get_ground_truth_from_json
from packages.getmetrics import evaluate_and_metrics
import os

In [2]:
audio_folder = r"D:\Coding\PII Evaluation\Audio output"
ground_truth_folder = r"D:\Coding\PII Evaluation\Annotations output"

In [3]:
import os
import sys

def get_files_from_folder(folder_path):
    """
    Gets a list of all files in the specified folder (non-recursive).

    Args:
        folder_path (str): The path to the folder.

    Returns:
        list: A list of filenames, or an empty list if the path is invalid.
    """
    try:
        # Get all entries in the directory
        all_entries = os.listdir(folder_path)
        
        # Filter the list to include only files, not directories
        files_only = [f for f in all_entries if os.path.isfile(os.path.join(folder_path, f))]
        
        return files_only
    except FileNotFoundError:
        print(f"Error: The folder '{folder_path}' was not found.", file=sys.stderr)
        return []
    except Exception as e:
        print(f"An error occurred: {e}", file=sys.stderr)
        return []

In [4]:
audio_files = get_files_from_folder(audio_folder)

# 3. Print the results
if audio_files:
    print("--- Audio Files found in the folder ---")
    for file_name in audio_files:
        print(file_name)
else:
    print("No audio files were found or an error occurred.")

--- Audio Files found in the folder ---
4531643.wav
4541297.wav
4543041.wav
4544210.wav
4545753.wav
4547022.wav
4547936.wav
4548579.wav
4549669.wav
4549891.wav
4550787.wav
4551012.wav
5369770.wav
5371697.wav
5373349.wav
5391472.wav
5392044.wav
5394622.wav
5396166.wav
5397141.wav


In [5]:
json_files = get_files_from_folder(r"D:\Coding\PII Evaluation\Annotations output")

# 3. Print the results
if json_files:
    print("--- JSON Files found in the folder ---")
    for file_name in json_files:
        print(file_name)
else:
    print("No JSON files were found or an error occurred.")

--- JSON Files found in the folder ---
DS-19768.DP-0bbad67112dd4f3d8ae0a2b96d14184a.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-17f0bced19c04ea1b61fd1d71411bb2b.TS-2025-08-09T13-30-37.899Z.json
DS-19768.DP-189131f00f2040f19a65d792398ca3d2.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-24fa4d88295749bb9989b2f7542fc3d6.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-38544210b4aa424388d9e24c249055f8.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-413a24bd3a834772b49e182545505690.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-421e734d22324e5eafbf0fb1a0306409.TS-2025-08-09T13-30-37.899Z.json
DS-19768.DP-4968a3326c2e41fb84c775c626ea0873.TS-2025-08-09T13-30-37.899Z.json
DS-19768.DP-4da7fd2df2274a4b819a8725d863ccc4.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-5ba72b70b56644babb2812c3641c94b8.TS-2025-08-09T13-30-37.899Z.json
DS-19768.DP-5f260e16414c4999b46329584b85a5c8.TS-2025-08-08T05-58-10.671Z.json
DS-19768.DP-74b0d98db00d4556bce48600d217d7c9.TS-2025-08-09T13-30-37.899Z.json
DS-19768.DP-7b13278222764

In [6]:
# transcript = transcribe_with_timestamps(audio_files)
# asr_text = [transcript['text']]
# pii_result = detect_pii_entities(asr_text)
# pii_words = []
# for doc in pii_result:
#     for entity in doc.entities:
#         if entity.confidence_score > 0.7 and entity.category in  ['Person', 'Organization', 'PhoneNumber', 'Address', 'Location', 'PhoneNumber', 'BankAccountNumber']:
#             pii_words.append(entity.text)
# pii_segments = map_pii_to_timestamps(pii_words, transcript['chunks'])
# ground_truth = get_ground_truth_from_json(json_files)
# sorted_ranges = sorted(pii_segments, key=lambda x: x)  # get the timestamps in a list
# metrics = evaluate_and_metrics(ground_truth, sorted_ranges)
# print(f"Metrics: {metrics}")

In [7]:
import json
import csv

In [ ]:
output_csv_path = "Performance_Metrics.csv"
csv_headers = ['Audiofile_name', 'Total_PII_Detected', 'True Positive', 'False Positive', 'False Negative', 'True Negative', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'F2 Score']

file_exists = os.path.exists(output_csv_path)

all_file_metrics = []

try:
    with open(output_csv_path, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=csv_headers)

        if not file_exists:
            writer.writeheader()

        for json_filename in os.listdir(ground_truth_folder):
            if not json_filename.endswith(".json"):
                continue

            ground_truth_path = os.path.join(ground_truth_folder, json_filename)
            
            try:
                with open(ground_truth_path, 'r') as f:
                    data = json.load(f)

                ground_truth_segments = []
                if "data" in data and "segments" in data["data"]:
                    for segment in data["data"]["segments"]:
                        ground_truth_segments.append((segment["start"] / 1000, segment["end"] / 1000))
                
                audio_filename = data["file_map"].get(data["files"]["audio"])
                if not audio_filename: continue

                input_audio_path = os.path.join(audio_folder, audio_filename)
                if not os.path.exists(input_audio_path): continue
                    
                print(f"\n--- Processing pair: {json_filename} -> {audio_filename} ---")

                transcript = transcribe_with_timestamps(input_audio_path)
                asr_text = [transcript['text']]
                pii_result = detect_pii_entities(asr_text)
                pii_words = [entity.text for doc in pii_result for entity in doc.entities if entity.confidence_score > 0.7 and entity.category in ['Person', 'Organization', 'PhoneNumber', 'Address', 'Location', 'BankAccountNumber']]
                pii_segments = map_pii_to_timestamps(pii_words, transcript['chunks'])
                sorted_ranges = sorted(pii_segments, key=lambda x: x)
                total_pii_detected = len(sorted_ranges)
                metrics = evaluate_and_metrics(ground_truth_segments, sorted_ranges)
                
                row_data = {'Audiofile_name': audio_filename, 'Total_PII_Detected': total_pii_detected, **metrics}
                writer.writerow(row_data)
                csvfile.flush()
                print(f"Metrics for {audio_filename}: {metrics}")
                all_file_metrics.append(row_data)

            except (json.JSONDecodeError, KeyError, FileNotFoundError) as e:
                print(f"Error processing {json_filename}: {e}. Skipping.", file=sys.stderr)

except IOError:
    print(f"Error: Could not write to CSV file at '{output_csv_path}'. Check permissions.", file=sys.stderr)

print(f"\nProcessing complete. New results appended to {output_csv_path}")



--- Processing pair: DS-19768.DP-0bbad67112dd4f3d8ae0a2b96d14184a.TS-2025-08-08T05-58-10.671Z.json -> 4547022.wav ---
Metrics for 4547022.wav: {'Accuracy': 0.81, 'Precision': 0.33, 'Recall': 0.4, 'F1 Score': 0.36, 'F2 Score': 0.38, 'True Positive': 2, 'False Positive': 4, 'False Negative': 3, 'True Negative': 27}

--- Processing pair: DS-19768.DP-17f0bced19c04ea1b61fd1d71411bb2b.TS-2025-08-09T13-30-37.899Z.json -> 5397141.wav ---
Metrics for 5397141.wav: {'Accuracy': 0.87, 'Precision': 0.23, 'Recall': 0.83, 'F1 Score': 0.36, 'F2 Score': 0.55, 'True Positive': 5, 'False Positive': 17, 'False Negative': 1, 'True Negative': 112}

--- Processing pair: DS-19768.DP-189131f00f2040f19a65d792398ca3d2.TS-2025-08-08T05-58-10.671Z.json -> 5373349.wav ---
Metrics for 5373349.wav: {'Accuracy': 0.0, 'Precision': 0, 'Recall': 0.0, 'F1 Score': 0, 'F2 Score': 0, 'True Positive': 0, 'False Positive': 0, 'False Negative': 7, 'True Negative': 0}

--- Processing pair: DS-19768.DP-24fa4d88295749bb9989b2f754

Error processing DS-19768.DP-24fa4d88295749bb9989b2f7542fc3d6.TS-2025-08-08T05-58-10.671Z.json: 'text'. Skipping.



--- Processing pair: DS-19768.DP-38544210b4aa424388d9e24c249055f8.TS-2025-08-08T05-58-10.671Z.json -> 5371697.wav ---
